# Challenge 2: Online Shopper Purchasing Intention

Predicting whether an online shopping session results in a purchase (`Revenue`).
Target is imbalanced, so F1 score on the positive class is the primary metric.

## Task 1: Data Preparation

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

In [2]:
train = pd.read_csv("challenge2_train.csv")
test = pd.read_csv("challenge2_test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

display(train.head())

Training data shape: (9247, 19)
Test data shape: (3083, 18)


,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


In [3]:
train.info()

print("\nMissing values in train:", train.isna().sum().sum())
print("Missing values in test:", test.isna().sum().sum())
print("Duplicate IDs in train:", train["ID"].duplicated().sum())
print("ID overlap between train and test:", len(set(train["ID"]) & set(test["ID"])))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   object 
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                  

In [4]:
train.info()

print("\nMissing values in train:", train.isna().sum().sum())
print("Missing values in test:", test.isna().sum().sum())
print("Duplicate IDs in train:", train["ID"].duplicated().sum())
print("ID overlap between train and test:", len(set(train["ID"]) & set(test["ID"])))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   object 
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                  

In [5]:
display(train.describe().T)

,count,mean,std,min,25%,50%,75%,max
ID,9247.0,6133.869579,3560.547325,1.0,3059.500000,6103.000000,9235.500000,12330.000000
Administrative,9247.0,2.314372,3.319935,0.0,0.000000,1.000000,4.000000,27.000000
Administrative_Duration,9247.0,81.168776,179.593423,0.0,0.000000,8.000000,92.550000,3398.750000
Informational,9247.0,0.502433,1.272332,0.0,0.000000,0.000000,0.000000,24.000000
Informational_Duration,9247.0,34.517405,141.310276,0.0,0.000000,0.000000,0.000000,2549.375000
ProductRelated,9247.0,31.358170,43.539969,0.0,7.000000,18.000000,38.000000,705.000000
ProductRelated_Duration,9247.0,1181.847920,1908.101262,0.0,181.416667,598.250000,1464.986002,63973.522230
BounceRates,9247.0,0.022268,0.048680,0.0,0.000000,0.003030,0.016834,0.200000
ExitRates,9247.0,0.043231,0.048835,0.0,0.014286,0.025088,0.050000,0.200000
PageValues,9247.0,5.983551,18.753351,0.0,0.000000,0.000000,0.000000,361.763742


In [6]:
print(train["Revenue"].value_counts())
print("\nPositive class rate:", round(train["Revenue"].mean(), 4))
print("Majority-class baseline accuracy:", round(1 - train["Revenue"].mean(), 4))

Revenue
False    7816
True     1431
Name: count, dtype: int64

Positive class rate: 0.1548
Majority-class baseline accuracy: 0.8452


### Class Imbalance

The target is imbalanced: only 15.5% of sessions end in a purchase. A model that
predicts `False` for every session achieves 84.5% accuracy while producing an F1
score of 0 for the positive class. Accuracy is therefore not a usable metric here,
and F1 on `Revenue=True` is used as the primary score throughout.

In [7]:
print("Conversion rate by Month:")
print(train.groupby("Month")["Revenue"].agg(["mean", "count"])
      .sort_values("mean", ascending=False).round(3))

print("\nConversion rate by VisitorType:")
print(train.groupby("VisitorType")["Revenue"].agg(["mean", "count"]).round(3))

print("\nConversion rate by Weekend:")
print(train.groupby("Weekend")["Revenue"].agg(["mean", "count"]).round(3))

Conversion rate by Month:
        mean  count
Month              
Nov    0.248   2213
Oct    0.224    415
Sep    0.197    345
Aug    0.185    319
Jul    0.135    319
Dec    0.132   1296
May    0.108   2550
Mar    0.104   1445
June   0.103    204
Feb    0.014    141

Conversion rate by VisitorType:
                    mean  count
VisitorType                    
New_Visitor        0.247   1265
Other              0.181     72
Returning_Visitor  0.140   7910

Conversion rate by Weekend:
          mean  count
Weekend              
False    0.148   7116
True     0.176   2131


In [8]:
print("Correlation with Revenue:")
print(train.select_dtypes(include=[np.number]).drop(columns=["ID"])
      .assign(Revenue=train["Revenue"].astype(int))
      .corr()["Revenue"].drop("Revenue").sort_values(ascending=False).round(3))

print("\nPageValues == 0 share:", round((train["PageValues"] == 0).mean(), 3))
print("Conversion when PageValues == 0:",
      round(train.loc[train["PageValues"] == 0, "Revenue"].mean(), 4))
print("Conversion when PageValues  > 0:",
      round(train.loc[train["PageValues"] > 0, "Revenue"].mean(), 4))

Correlation with Revenue:
PageValues                 0.504
ProductRelated             0.162
ProductRelated_Duration    0.151
Administrative             0.143
Administrative_Duration    0.100
Informational              0.095
Informational_Duration     0.067
Browser                    0.020
TrafficType                0.000
Region                    -0.007
OperatingSystems          -0.010
SpecialDay                -0.089
BounceRates               -0.154
ExitRates                 -0.214
Name: Revenue, dtype: float64

PageValues == 0 share: 0.777
Conversion when PageValues == 0: 0.0363
Conversion when PageValues  > 0: 0.5685


### Key Findings

`PageValues` is by far the strongest single predictor (correlation 0.504, roughly
triple any other feature). It is zero for 77.7% of sessions, and that split is close
to a natural decision boundary: sessions with `PageValues == 0` convert at 3.6%,
while sessions with `PageValues > 0` convert at 56.9%.

`ExitRates` (-0.214) and `BounceRates` (-0.154) are the next strongest signals, both
negative as expected. `Month` carries real information, ranging from 24.8% conversion
in November down to 1.4% in February. `VisitorType` shows new visitors converting at
24.7% versus 14.0% for returning visitors.

In [9]:
X = train.drop(columns=["ID", "Revenue"])
y = train["Revenue"].astype(int)

X_test = test.drop(columns=["ID"])
test_ids = test["ID"]

print("X shape:", X.shape, "| y positive rate:", round(y.mean(), 4))
print("X_test shape:", X_test.shape)

X shape: (9247, 17) | y positive rate: 0.1548
X_test shape: (3083, 17)


### Feature Types

`OperatingSystems`, `Browser`, `Region`, and `TrafficType` are stored as integers but
are nominal identifiers, not ordered quantities. Treating them as numeric would imply
that Browser 13 is greater than Browser 1, a relationship that does not exist. Their
correlations with the target (0.020, -0.010, -0.007, and 0.000 respectively) confirm
there is no usable linear ordering. They are therefore one-hot encoded alongside
`Month` and `VisitorType`.

Feature lists are defined explicitly rather than inferred from dtype, since dtype
inference would misclassify all four.

In [10]:
numeric_features = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]

categorical_features = [
    "Month", "VisitorType",
    "OperatingSystems", "Browser", "Region", "TrafficType",
]

binary_features = ["Weekend"]

assert set(numeric_features + categorical_features + binary_features) == set(X.columns)
print("All", X.shape[1], "features assigned.")

All 17 features assigned.


### Preprocessing

Numeric features are standardized. Scaling is required here, not optional: K-Nearest
Neighbors and Support Vector Machines are distance-based and would be dominated by
whichever feature has the largest raw range, and Logistic Regression converges poorly
without it. Tree-based models are unaffected by monotonic rescaling, so a single
shared pipeline serves all seven required models.

`handle_unknown="ignore"` guards against a rare category level being absent from an
individual cross-validation fold.

In [11]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("bin", "passthrough", binary_features),
])

print("Output shape after preprocessing:", preprocessor.fit_transform(X).shape)

Output shape after preprocessing: (9247, 74)


## Task 2: Validation Strategy

We use 5-fold **stratified** cross-validation. Stratification preserves the 15.5%
positive rate within every fold; plain K-fold can produce folds whose class balance
drifts enough to make F1 scores incomparable across models.

F1 on the positive class is the primary metric, matching the competition. Precision
and recall are reported alongside it because F1 alone hides which side a model is
failing on, and ROC AUC is included as a threshold-independent check. Accuracy is
reported only to demonstrate how misleading it is at this class balance.

Train-fold and validation-fold F1 are both reported so the gap between them exposes
overfitting. The same `StratifiedKFold` object is reused for every model and carries
forward into GridSearchCV during tuning.

Note for the tuning stage: `class_weight="balanced"` is available on Logistic
Regression, SVM, Decision Tree, and Random Forest, and is likely to improve F1
meaningfully at this class balance. It belongs in the hyperparameter grid rather
than in preprocessing.

### Development and Holdout Split

The training data is split into a development set (85%) and a holdout set (15%),
stratified to preserve the class balance in both. All cross-validation, hyperparameter
tuning, and model comparison use the development set only.

The holdout is evaluated exactly once, during final model selection, as a check on
whether cross-validation results held up on data that played no part in tuning. At
roughly 215 positive cases it is large enough to catch a model that benefited from
fold-specific noise, but not large enough to serve as the selection metric itself.

In [12]:
from sklearn.model_selection import train_test_split

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

print("Development set:", X_dev.shape, "| positive rate:", round(y_dev.mean(), 4))
print("Holdout set:    ", X_holdout.shape, "| positive rate:", round(y_holdout.mean(), 4))

Development set: (7859, 17) | positive rate: 0.1547
Holdout set:     (1388, 17) | positive rate: 0.1549


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

SCORING = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
}

def cv_evaluate(name, model):
    results = cross_validate(
        model, X_dev, y_dev,
        cv=cv,
        scoring=SCORING,
        return_train_score=True,
        n_jobs=-1,
    )
    return {
        "Model": name,
        "Train F1": results["train_f1"].mean(),
        "CV F1": results["test_f1"].mean(),
        "F1 Std": results["test_f1"].std(),
        "Gap": results["train_f1"].mean() - results["test_f1"].mean(),
        "Precision": results["test_precision"].mean(),
        "Recall": results["test_recall"].mean(),
        "ROC AUC": results["test_roc_auc"].mean(),
        "Accuracy": results["test_accuracy"].mean(),
        "Fit time (s)": results["fit_time"].mean(),
    }

In [14]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

def cv_confusion(name, model):
    preds = cross_val_predict(model, X_dev, y_dev, cv=cv, n_jobs=-1)   # ← changed
    print(f"--- {name} ---")
    print(confusion_matrix(y_dev, preds))                               # ← changed
    print(classification_report(y_dev, preds, target_names=["No Purchase", "Purchase"]))
    return preds

### Decision Threshold

Classifiers default to a 0.5 probability threshold, which is rarely F1-optimal on
imbalanced data. Because F1 on the positive class is the competition metric, the
threshold is treated as a tunable parameter: cross-validated predicted probabilities
are swept across the precision-recall curve and the threshold maximizing F1 is
selected. The threshold is chosen on development-set validation folds only.

In [15]:
from sklearn.metrics import precision_recall_curve, f1_score

def tune_threshold(name, model):
    proba = cross_val_predict(
        model, X_dev, y_dev, cv=cv, method="predict_proba", n_jobs=-1
    )[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_dev, proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    best = f1[:-1].argmax()

    return {
        "Model": name,
        "F1 @ 0.50": f1_score(y_dev, proba >= 0.5),
        "Best threshold": thresholds[best],
        "F1 @ best": f1[best],
    }

### Framework Check

A baseline Logistic Regression confirms the preprocessing and evaluation framework
run end to end, and provides a reference row for the model comparison in Task 3.

In [16]:
from sklearn.linear_model import LogisticRegression

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

display(pd.DataFrame([cv_evaluate("Logistic Regression (baseline)", baseline)]).round(3))
display(pd.DataFrame([tune_threshold("Logistic Regression (baseline)", baseline)]).round(3))

,Model,Train F1,CV F1,F1 Std,Gap,Precision,Recall,ROC AUC,Accuracy,Fit time (s)
0,Logistic Regression (baseline),0.534,0.522,0.027,0.012,0.756,0.4,0.895,0.887,0.214


,Model,F1 @ 0.50,Best threshold,F1 @ best
0,Logistic Regression (baseline),0.523,0.233,0.633
